In [7]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import genextreme as gev
import geopandas as gpd
import regionmask
import xarray as xr

from unseen import eva
from unseen import fileio

import utils

In [2]:
#import importlib

In [3]:
#importlib.reload(utils)

In [3]:
# optional parameters
# (this cell is tagged "parameters")

metric = 'txx'
location = 'Katherine'
model_dict = {
    'BCC-CSM2-MR': 'tab:blue',
    'CAFE': 'tab:orange',
    'CMCC-CM2-SR5': 'tab:green',
    'CanESM5': 'tab:red',
    'EC-Earth3': 'tab:purple',
    'IPSL-CM6A-LR': 'tab:brown',
    'MIROC6': 'tab:pink',
    'MPI-ESM1-2-HR': 'tab:grey',
    'MRI-ESM2-0': 'tab:olive',
    'NorCPM1': 'tab:cyan',
}  

In [4]:
assert "location" in locals(), "Must provide a location name"
assert "metric" in locals(), "Must provide a metric (rx1day or txx)"
assert "model_dict" in locals(), "Must provide a model dict"

In [5]:
def extract_closest_row(df, return_period):
    """Extract the row from df closest to the return period."""

    differences = np.array(np.abs(return_df.index - 100))
    index = np.argmin(differences)

    return df.iloc[index]

In [8]:
infile = '/g/data/xv83/unseen-projects/outputs/bias/data/rx1day_AGCD-CSIRO_1901-2024_annual-jul-to-jun_AUS300i.nc'
ds = fileio.open_dataset(infile)
overlap_threshold = 0.01
shape_gpd = gpd.read_file('/g/data/ia39/aus-ref-clim-data-nci/shapefiles/data/australia/australia.shp')
shape_rgm = regionmask.from_geopandas(
    shape_gpd,
    names="AUS_NAME21",
    abbrevs="AUS_CODE21",
    name="australia"
)
frac = shape_rgm.mask_3D_frac_approx(ds)
mask = frac.sel(region=0) >= overlap_threshold

In [10]:
mask = mask.isel(lat=slice(0,2))

In [ ]:
## TODO: Build xarray data arrays for output netCDF

In [13]:
return_level = 100
nlat, nlon = mask.shape
G_array = np.zeros(mask.shape)
M_array = np.zeros(mask.shape)
B_array = np.zeros(mask.shape)
TO_array = np.zeros(mask.shape)
MMM_array = np.zeros(mask.shape)
obs_array = np.zeros(mask.shape)
for lat_index in range(nlat):
    for lon_index in range(nlon):
        print(lat_index, lon_index)
        include_point = mask.isel({'lat': lat_index, 'lon': lon_index})
        if include_point:
            return_df, gev_spread_df = utils.get_return_values(metric, [lat_index, lon_index], model_dict, similarity_check=True)
            try:
                obs, MMM, uncertainty = utils.uncertainty_breakdown(return_df, gev_spread_df)
                G, M, B, T, O = uncertainty
                Gs = extract_closest_row(G, return_level)
                Ms = extract_closest_row(M, return_level)
                Bs = extract_closest_row(B, return_level)
                Ts = extract_closest_row(T, return_level)
                Os = extract_closest_row(O, return_level)
                MMMs = extract_closest_row(MMM, return_level)
                obss = extract_closest_row(obs, return_level)
                GMBs = Gs + Ms + Bs
                Gs_pct = (Gs / GMBs) * 100
                Ms_pct = (Ms / GMBs) * 100
                Bs_pct = (Bs / GMBs) * 100
                TO_ratio = Ts / Os
            except ValueError:
                Gs_pct = np.nan
                Ms_pct = np.nan
                Bs_pct = np.nan
                TO_ratio = np.nan
                obss = np.nan
                MMMs = np.nan
        else:
            Gs_pct = np.nan
            Ms_pct = np.nan
            Bs_pct = np.nan
            TO_ratio = np.nan
            obss = np.nan
            MMMs = np.nan
            
        G_array[lat_index, lon_index] = Gs_pct
        M_array[lat_index, lon_index] = Ms_pct
        B_array[lat_index, lon_index] = Bs_pct
        TO_array[lat_index, lon_index] = TO_ratio
        MMM_array[lat_index, lon_index] = MMMs
        obs_array[lat_index, lon_index] = obss

## TODO: Include an array of the number of passing models.

0 0
0 1
0 2
0 3
0 4
0 5
0 6
0 7
0 8
0 9
0 10
start: BCC-CSM2-MR
start: CAFE
start: CMCC-CM2-SR5
start: CanESM5
start: EC-Earth3
start: IPSL-CM6A-LR
start: MIROC6
start: MPI-ESM1-2-HR
start: MRI-ESM2-0
start: NorCPM1
0 11
start: BCC-CSM2-MR
end: BCC-CSM2-MR
start: CAFE
end: CAFE
start: CMCC-CM2-SR5
end: CMCC-CM2-SR5
start: CanESM5
start: EC-Earth3
end: EC-Earth3
start: IPSL-CM6A-LR
start: MIROC6
end: MIROC6
start: MPI-ESM1-2-HR
start: MRI-ESM2-0
end: MRI-ESM2-0
start: NorCPM1
0 12
start: BCC-CSM2-MR
start: CAFE
start: CMCC-CM2-SR5
start: CanESM5
start: EC-Earth3
start: IPSL-CM6A-LR
start: MIROC6
end: MIROC6
start: MPI-ESM1-2-HR
start: MRI-ESM2-0
start: NorCPM1
0 13
1 0
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
start: BCC-CSM2-MR
start: CAFE
start: CMCC-CM2-SR5
end: CMCC-CM2-SR5
start: CanESM5
end: CanESM5
start: EC-Earth3
start: IPSL-CM6A-LR
start: MIROC6
end: MIROC6
start: MPI-ESM1-2-HR
start: MRI-ESM2-0
end: MRI-ESM2-0
start: NorCPM1
1 10
start: BCC-CSM2-MR
start: CAFE
start: CMCC-CM2-SR5
e

In [11]:
#print(f"GEV uncertainty: {Gs_pct:.1f}%")
#print(f"Model uncertainty: {Ms_pct:.1f}%")
#print(f"Bias correction uncertainty: {Bs_pct:.1f}%")
#print(f"Model to obs uncertainty ratio: {TO_ratio:.1f}")

GEV uncertainty: 4.5%
Model uncertainty: 65.8%
Bias correction uncertainty: 29.7%
Model to obs uncertainty ratio: 2.0
